# Pyre — HuggingFace Baseline Evaluation

Evaluates an **un-fine-tuned** (or fine-tuned) HuggingFace model across all 3 Pyre difficulty levels
using the same episode-running logic as `evals.py`.

**Workflow:**
1. Edit **Cell 2 (Config)** with your model path, server URL, and eval settings
2. Run all cells top-to-bottom (`Run All`)
3. To change any parameter — update Cell 2 and re-run from Cell 7 onward

## Cell 1 — Imports

In [1]:
import csv
import json
import logging
import re
import textwrap
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests
from langchain_core.language_models.chat_models import SimpleChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from pydantic import PrivateAttr

from pyre_env import PyreEnv, PyreAction

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("evals_hf")
print("Imports OK")

Imports OK


## Cell 2 — Config  ✏️ Edit this cell to change eval settings

In [2]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen3-1.7B"   # HF model ID or local path / adapter dir
LOAD_4BIT       = False                # True → bitsandbytes 4-bit quant (low VRAM)
TEMPERATURE     = 0.3                  # 0 = greedy decoding
MAX_NEW_TOKENS  = 512                  # max tokens per model response

# ── Environment ───────────────────────────────────────────────────────────────
ENV_URL         = "http://localhost:8000"  # running Pyre server

# ── Difficulties ──────────────────────────────────────────────────────────────
# Set to None to run all 3 levels, or list specific ones:
#   ["easy", "medium", "hard"]
DIFFICULTIES_TO_RUN = None

# ── Seeds ─────────────────────────────────────────────────────────────────────
NUM_SEEDS       = 3     # episodes per difficulty level
SEED_START      = 1     # first seed value

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = "./outputs/hf_evals"  # CSV + debug traces saved here
VERBOSE         = False                  # True → per-step DEBUG logs

# ── Derived (do not edit) ─────────────────────────────────────────────────────
if VERBOSE:
    log.setLevel(logging.DEBUG)

SEEDS = list(range(SEED_START, SEED_START + NUM_SEEDS))
print(f"Model      : {MODEL_ID}")
print(f"4-bit      : {LOAD_4BIT}")
print(f"Temperature: {TEMPERATURE}")
print(f"Levels     : {DIFFICULTIES_TO_RUN or 'all 3 (easy, medium, hard)'}")
print(f"Seeds      : {SEEDS}")
print(f"Output dir : {OUTPUT_DIR}")

Model      : Qwen/Qwen3-1.7B
4-bit      : False
Temperature: 0.3
Levels     : all 3 (easy, medium, hard)
Seeds      : [1, 2, 3]
Output dir : ./outputs/hf_evals


## Cell 3 — Difficulty Registry & System Prompt

In [3]:
ALL_DIFFICULTIES: List[Dict[str, Any]] = [
    {"difficulty": "easy",   "max_steps": 200, "description": "1 fire source · slow spread · calm wind · high humidity"},
    {"difficulty": "medium", "max_steps": 150, "description": "2–4 fire sources · moderate spread · any wind · moderate humidity"},
    {"difficulty": "hard",   "max_steps": 100, "description": "3–5 fire sources · fast spread · always windy · low humidity"},
]

# Filter down to DIFFICULTIES_TO_RUN if set in Config cell
difficulties_to_run = (
    [d for d in ALL_DIFFICULTIES if d["difficulty"] in DIFFICULTIES_TO_RUN]
    if DIFFICULTIES_TO_RUN else ALL_DIFFICULTIES
)
if not difficulties_to_run:
    raise ValueError(f"No matching difficulties. Available: {[d['difficulty'] for d in ALL_DIFFICULTIES]}")

SYSTEM_PROMPT = textwrap.dedent("""
You are an agent trapped inside a burning building.
Your goal is to navigate to an EXIT before your health reaches zero or time runs out.

ENVIRONMENT RULES
- You have partial vision: you cannot see through walls or dense smoke.
- Fire and smoke spread each step — do NOT linger in hazardous areas.
- Closing a door adjacent to active fire slows its spread (strategic move).
- Your health drains faster in moderate/heavy smoke and on fire cells.
- Exits may be BLOCKED if fire burns directly on them — check available hints.

OUTPUT FORMAT (STRICT)
You MUST reason inside <think>...</think> tags first, then emit EXACTLY ONE JSON object.
Output NOTHING else — no extra text, no markdown fences, no second JSON block.

<think>
Brief reasoning: what can I see, where is danger, what is the safest next move?
</think>
{"action": "move", "direction": "north"}

AVAILABLE ACTIONS
- move  : {"action": "move", "direction": "north|south|east|west"}
- look  : {"action": "look", "direction": "north|south|east|west"}   ← scan 5 cells ahead
- door  : {"action": "door", "target_id": "door_X", "door_state": "open|close"}
- wait  : {"action": "wait"}

REWARD SIGNAL (shown in history after each step)
- Positive reward  → you moved closer to an exit or played a smart move.
- Negative reward  → you moved into danger, stalled, or wasted a step.
- Use the reward trend to judge if your current direction is working.

STRATEGY TIPS
- Use `look` to scout a direction before entering an unknown corridor.
- Closing a door between you and fire buys time; re-open when clear.
- Prefer moves that increase reward — progress toward the exit is rewarded.
- If smoke is heavy, back away; your health drains fast in thick smoke.
- Door IDs (e.g. door_3) appear in the Visible objects list — use them with the door action.
""").strip()

print(f"Difficulties to evaluate : {[d['difficulty'] for d in difficulties_to_run]}")
print(f"Total episodes           : {len(difficulties_to_run) * len(SEEDS)}")

Difficulties to evaluate : ['easy', 'medium', 'hard']
Total episodes           : 9


## Cell 4 — HuggingFace Chat Model Wrapper

In [4]:
class HFChatModel(SimpleChatModel):
    """
    Wraps a locally-loaded HuggingFace causal-LM as a LangChain chat model.
    Uses the model loading + chat-template logic compatible with Qwen3 and similar models.
    """

    model_id: str
    temperature: float = 0.3
    max_new_tokens: int = 512

    _model: Any = PrivateAttr(default=None)
    _tokenizer: Any = PrivateAttr(default=None)

    def load(self, load_4bit: bool = False) -> "HFChatModel":
        """Load model + tokenizer. Call once after construction."""
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        log.info("Loading tokenizer from %s …", self.model_id)
        tokenizer = AutoTokenizer.from_pretrained(self.model_id, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        self._tokenizer = tokenizer

        log.info("Loading model from %s (4bit=%s) …", self.model_id, load_4bit)
        model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            torch_dtype=torch.bfloat16 if not load_4bit else None,
            device_map="auto",
            load_in_4bit=load_4bit,
            trust_remote_code=True,
        )
        model.eval()
        self._model = model
        device_info = getattr(self._model, "hf_device_map", "auto")
        log.info("Model loaded. Device map: %s", device_info)
        return self

    def _call(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager=None,
        **kwargs,
    ) -> str:
        import torch

        # Convert LangChain messages → HF chat-template dicts
        conversation: List[Dict[str, str]] = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                conversation.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                conversation.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                conversation.append({"role": "assistant", "content": msg.content})

        # Apply chat template — try with enable_thinking first (Qwen3), fall back for others
        try:
            prompt_text = self._tokenizer.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except TypeError:
            prompt_text = self._tokenizer.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=True,
            )

        inputs  = self._tokenizer(prompt_text, return_tensors="pt").to(self._model.device)
        in_len  = inputs["input_ids"].shape[1]

        gen_kwargs: Dict[str, Any] = dict(
            max_new_tokens=self.max_new_tokens,
            pad_token_id=self._tokenizer.eos_token_id,
            eos_token_id=self._tokenizer.eos_token_id,
        )
        if self.temperature > 0:
            gen_kwargs.update(do_sample=True, temperature=self.temperature)
        else:
            gen_kwargs["do_sample"] = False

        with torch.inference_mode():
            output_ids = self._model.generate(**inputs, **gen_kwargs)

        new_tokens = output_ids[0][in_len:]
        return self._tokenizer.decode(new_tokens, skip_special_tokens=True)

    @property
    def _llm_type(self) -> str:
        return "hf-chat-model"

print("HFChatModel class defined.")

HFChatModel class defined.


## Cell 5 — Prompt Builder & Action Parser

In [5]:
def _build_user_message(obs: Dict[str, Any], history: List[str]) -> str:
    """Convert a raw observation dict + history into the LLM user message."""
    narrative = obs.get("narrative", "(no narrative)")
    # Strip the "Available actions:" line — the system prompt already covers this.
    narrative = re.sub(r"\nAvailable actions:.*$", "", narrative, flags=re.MULTILINE)

    health       = obs.get("agent_health", "?")
    health_st    = obs.get("health_status", "?")
    location     = obs.get("location_label", "?")
    smoke        = obs.get("smoke_level", "none")
    fire_vis     = obs.get("fire_visible", False)
    fire_dir     = obs.get("fire_direction") or "none"
    wind         = obs.get("wind_dir", "CALM")
    elapsed      = obs.get("elapsed_steps", 0)
    blocked_exits = obs.get("blocked_exit_ids", [])
    visible_objs  = obs.get("visible_objects", [])
    audible       = obs.get("audible_signals", [])

    status_line = (
        f"Health: {health:.1f} | Status: {health_st} | Location: {location}\n"
        f"Smoke: {smoke} | Fire visible: {fire_vis}"
        + (f" (direction: {fire_dir})" if fire_vis else "")
        + f"\nWind: {wind} | Steps elapsed: {elapsed}"
    )
    if blocked_exits:
        status_line += f"\nBLOCKED exits (fire on them): {', '.join(blocked_exits)}"
    if visible_objs:
        obj_strs = [
            f"{o.get('type','?')} '{o.get('id','?')}' {o.get('relative_pos','?')}"
            + (f" [{o.get('state','')}]" if o.get("state") else "")
            for o in visible_objs
        ]
        status_line += f"\nVisible objects: {'; '.join(obj_strs)}"
    if audible:
        status_line += f"\nSounds: {'; '.join(audible)}"

    history_str = ""
    if history:
        recent = history[-8:]  # last 8 steps to stay within context limits
        history_str = (
            "=== RECENT ACTION HISTORY (action → feedback → reward → health) ===\n"
            + "\n".join(recent) + "\n\n"
        )

    return (
        f"=== CURRENT OBSERVATION ===\n{narrative}\n\n"
        f"=== STATUS ===\n{status_line}\n\n"
        + history_str
        + "What is your next action? Respond with <think>...</think> then a single JSON action."
    )


_VALID_ACTIONS     = {"move", "door", "look", "wait"}
_VALID_DIRECTIONS  = {"north", "south", "east", "west"}
_VALID_DOOR_STATES = {"open", "close"}
_FALLBACK_ACTION   = {"action": "wait"}


def _validate_pyre_action(blob: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Return a sanitised action dict or None if the blob is unusable."""
    action = blob.get("action", "").strip().lower()
    if action not in _VALID_ACTIONS:
        return None

    out: Dict[str, Any] = {"action": action}

    if action in ("move", "look"):
        direction = str(blob.get("direction", "")).strip().lower()
        if direction not in _VALID_DIRECTIONS:
            return None
        out["direction"] = direction

    elif action == "door":
        tid = blob.get("target_id", "") or blob.get("target", "")
        ds  = str(blob.get("door_state", "")).strip().lower()
        if not tid or ds not in _VALID_DOOR_STATES:
            return None
        out["target_id"]  = str(tid)
        out["door_state"] = ds

    return out


def _parse_pyre_action(text: str) -> Tuple[Dict[str, Any], float]:
    """
    Extract a Pyre action from raw LLM text.

    Returns (action_dict, format_score) where format_score reflects output quality:
      1.0  — valid JSON + <think> tags
      0.7  — valid JSON, no <think>
      0.4  — partial JSON rescued via regex
      0.1  — action keyword found in raw text (last resort)
      0.0  — completely unparseable → {"action": "wait"} fallback
    """
    has_think = "<think>" in text and "</think>" in text

    # Level 1: well-formed JSON
    start = text.find("{")
    end   = text.rfind("}")
    if start != -1 and end > start:
        try:
            blob = json.loads(text[start:end + 1])
            if isinstance(blob, dict):
                action = _validate_pyre_action(blob)
                if action is not None:
                    return action, (1.0 if has_think else 0.7)
        except json.JSONDecodeError:
            pass

    # Level 2: regex — find the innermost {...} with "action" key
    for m in re.finditer(r'\{[^{}]+\}', text):
        try:
            blob = json.loads(m.group())
            if isinstance(blob, dict) and "action" in blob:
                action = _validate_pyre_action(blob)
                if action is not None:
                    return action, 0.4
        except json.JSONDecodeError:
            continue

    # Level 3: bare keyword extraction
    lower = text.lower()
    for d in _VALID_DIRECTIONS:
        if f"move {d}" in lower:
            return {"action": "move", "direction": d}, 0.1
    for d in _VALID_DIRECTIONS:
        if f"look {d}" in lower:
            return {"action": "look", "direction": d}, 0.1
    door_m = re.search(r'door[_\s]*([\w]+)', lower)
    if door_m:
        tid = door_m.group(1)
        ds  = "close" if "clos" in lower else "open"
        return {"action": "door", "target_id": tid, "door_state": ds}, 0.1
    if "wait" in lower:
        return {"action": "wait"}, 0.1

    # Level 4: total parse failure
    return dict(_FALLBACK_ACTION), 0.0


print("Prompt builder and action parser defined.")

Prompt builder and action parser defined.


## Cell 6 — Episode Runner

In [6]:
def run_episode(
    llm: HFChatModel,
    difficulty: str,
    seed: int,
    max_steps: int,
    debug_dir: Optional[Path] = None,
) -> Dict[str, Any]:
    """Run one full episode using the PyreEnv sync client (WebSocket-based, stateful).

    The episode is self-contained: reset → loop(observe → LLM → act) → done.
    Uses PyreEnv.sync() so the WebSocket session persists across all steps.
    """
    step_rewards:      List[float] = []
    history:           List[str]   = []
    llm_responses_log: List[str]   = []
    steps_taken     = 0
    think_steps     = 0
    parsed_steps    = 0
    fmt_scores:     List[float] = []
    cause_of_end    = "timeout"
    final_health    = 0.0
    agent_evacuated = False

    try:
        with PyreEnv(base_url=ENV_URL).sync() as env:
            # ── Reset ─────────────────────────────────────────────────────────
            result = env.reset(difficulty=difficulty, seed=seed)
            obs    = result.observation   # PyreObservation
            done   = result.done

            if debug_dir is not None:
                try:
                    state_resp = requests.get(f"{ENV_URL}/state", timeout=10)
                    if state_resp.ok:
                        debug_dir.mkdir(parents=True, exist_ok=True)
                        (debug_dir / f"{difficulty}_seed{seed}_init_state.json").write_text(
                            json.dumps(state_resp.json(), indent=2)
                        )
                except Exception as exc:
                    log.warning("Could not save initial state: %s", exc)

            # ── Episode loop ─────────────────────────────────────────────────
            for _step in range(max_steps):
                if done:
                    break

                obs_dict = obs.model_dump()
                user_msg = _build_user_message(obs_dict, history)
                messages = [
                    SystemMessage(content=SYSTEM_PROMPT),
                    HumanMessage(content=user_msg),
                ]

                # ── LLM call ─────────────────────────────────────────────────
                try:
                    response        = llm.invoke(messages)
                    completion_text = response.content
                except Exception as exc:
                    log.warning("LLM call failed at step %d: %s", _step + 1, exc)
                    step_rewards.append(-0.20)
                    break

                llm_responses_log.append(f"## Step {_step + 1}\n{completion_text}\n")

                has_think = "<think>" in completion_text and "</think>" in completion_text
                if has_think:
                    think_steps += 1
                steps_taken += 1

                # ── Parse action ─────────────────────────────────────────────
                action_dict, fmt_score = _parse_pyre_action(completion_text)
                fmt_scores.append(fmt_score)

                if fmt_score > 0.0:
                    parsed_steps += 1
                else:
                    log.debug("  step=%d  UNPARSEABLE — using wait fallback", _step + 1)

                log.debug(
                    "  step=%d  fmt=%.1f  action=%s",
                    _step + 1, fmt_score, json.dumps(action_dict),
                )

                # ── Step the environment ──────────────────────────────────────
                try:
                    result      = env.step(PyreAction(**action_dict))
                    obs         = result.observation
                    step_reward = float(result.reward or 0.0)
                    done        = result.done
                except Exception as exc:
                    log.warning("Step failed at step %d: %s", _step + 1, exc)
                    step_rewards.append(-0.20)
                    break

                # Format penalty: imperfect parse loses up to -0.10
                fmt_penalty = (1.0 - fmt_score) * -0.10
                step_rewards.append(step_reward + fmt_penalty)

                feedback = obs.last_action_feedback or ""
                history.append(
                    f"Step {_step + 1}: {json.dumps(action_dict)}"
                    + (f"\n  → {feedback}" if feedback else "")
                    + f"\n  reward: {step_reward:+.3f}  health: {obs.agent_health:.1f}"
                )

            # ── Final state ───────────────────────────────────────────────────
            agent_evacuated = obs.agent_evacuated
            final_health    = float(obs.agent_health)

    except Exception as exc:
        log.error("Episode failed (difficulty=%s seed=%d): %s", difficulty, seed, exc)
        return {"difficulty": difficulty, "seed": seed, "error": str(exc), "evacuated": 0}

    # ── Determine cause of end ────────────────────────────────────────────────
    if agent_evacuated:
        cause_of_end = "evacuated"
    elif final_health <= 0.0:
        cause_of_end = "death"
    else:
        cause_of_end = "timeout"

    if debug_dir is not None and llm_responses_log:
        try:
            debug_dir.mkdir(parents=True, exist_ok=True)
            (debug_dir / f"{difficulty}_seed{seed}_llm_trace.md").write_text(
                "\n".join(llm_responses_log)
            )
        except Exception as exc:
            log.warning("Could not save LLM trace: %s", exc)

    total_reward     = sum(step_rewards)
    mean_step_reward = total_reward / max(len(step_rewards), 1)
    think_rate       = think_steps / max(steps_taken, 1)
    parse_rate       = parsed_steps / max(steps_taken, 1)
    fmt_score_avg    = sum(fmt_scores) / max(len(fmt_scores), 1)

    return {
        "difficulty":       difficulty,
        "seed":             seed,
        "evacuated":        int(agent_evacuated),
        "cause_of_end":     cause_of_end,
        "final_health":     round(final_health, 2),
        "total_reward":     round(total_reward, 4),
        "mean_step_reward": round(mean_step_reward, 4),
        "steps_taken":      steps_taken,
        "max_steps":        max_steps,
        "think_rate":       round(think_rate, 4),
        "parse_rate":       round(parse_rate, 4),
        "format_score_avg": round(fmt_score_avg, 4),
        "error":            None,
    }


print("Episode runner defined.")

Episode runner defined.


## Cell 7 — Server Health Check & Model Load

In [7]:
# Health check first — fail fast before waiting for the large model to load
try:
    health = requests.get(f"{ENV_URL}/health", timeout=5)
    health.raise_for_status()
    print(f"Server health check PASSED  ({ENV_URL})")
except Exception as exc:
    raise RuntimeError(f"Server not reachable at {ENV_URL}: {exc}")

# Build and load the HF model — this is the expensive step
llm = HFChatModel(
    model_id=MODEL_ID,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
).load(load_4bit=LOAD_4BIT)

print(f"\nModel ready: {MODEL_ID}")

RuntimeError: Server not reachable at http://localhost:8000: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))

## Cell 8 — Run Evaluation Loop

In [ ]:
results:   List[Dict[str, Any]] = []
debug_dir = Path(OUTPUT_DIR) / "debug"
total     = len(difficulties_to_run) * len(SEEDS)

for done_count, (diff_cfg, seed) in enumerate(
    [(d, s) for d in difficulties_to_run for s in SEEDS], start=1
):
    difficulty = diff_cfg["difficulty"]
    max_steps  = diff_cfg["max_steps"]

    log.info("[%d/%d] difficulty=%-8s seed=%d", done_count, total, difficulty, seed)

    result = run_episode(
        llm=llm,
        difficulty=difficulty,
        seed=seed,
        max_steps=max_steps,
        debug_dir=debug_dir,
    )
    results.append(result)

    if result.get("error"):
        log.warning("  ERROR: %s", result["error"])
    else:
        evac_sym = "✓ EVACUATED" if result["evacuated"] else f"✗ {result['cause_of_end'].upper()}"
        log.info(
            "  %-14s  health=%5.1f  reward=%+.2f  steps=%d/%d  think=%d%%  parse=%d%%",
            evac_sym,
            result["final_health"],
            result["total_reward"],
            result["steps_taken"],
            max_steps,
            result["think_rate"] * 100,
            result["parse_rate"] * 100,
        )

print(f"\nAll {total} episodes complete.")

## Cell 9 — Summary Table & CSV Export

In [ ]:
def _avg(vals: List[float]) -> float:
    return round(sum(vals) / len(vals), 4) if vals else 0.0


by_diff: Dict[str, List[Dict]] = defaultdict(list)
for r in results:
    if not r.get("error"):
        by_diff[r["difficulty"]].append(r)

col    = "{:<10} {:>8} {:>10} {:>10} {:>11} {:>9} {:>9} {:>9}"
header = col.format(
    "Difficulty", "Evac%", "AvgHealth", "TotalRew",
    "MeanStpRew", "Steps/Max", "Think%", "Parse%",
)
sep = "=" * len(header)

print(f"\n{sep}")
print(f"  PYRE EVAL — model: {MODEL_ID}")
print(sep)
print(header)
print("-" * len(header))

diagnosis: List[str] = []

for cfg in ALL_DIFFICULTIES:
    diff     = cfg["difficulty"]
    max_steps = cfg["max_steps"]
    rows     = by_diff.get(diff, [])
    if not rows:
        print(col.format(diff, "n/a", "n/a", "n/a", "n/a", "n/a", "n/a", "n/a"))
        continue

    evac_rate     = _avg([r["evacuated"]         for r in rows])
    avg_health    = _avg([r["final_health"]       for r in rows])
    avg_total_rew = _avg([r["total_reward"]        for r in rows])
    avg_step_rew  = _avg([r["mean_step_reward"]    for r in rows])
    avg_steps     = _avg([r["steps_taken"]         for r in rows])
    avg_think     = _avg([r["think_rate"]          for r in rows])
    avg_parse     = _avg([r["parse_rate"]          for r in rows])

    cause_counts: Dict[str, int] = defaultdict(int)
    for r in rows:
        cause_counts[r["cause_of_end"]] += 1

    print(col.format(
        diff,
        f"{evac_rate * 100:.0f}%",
        f"{avg_health:.1f}",
        f"{avg_total_rew:+.2f}",
        f"{avg_step_rew:+.3f}",
        f"{avg_steps:.0f}/{max_steps}",
        f"{avg_think * 100:.0f}%",
        f"{avg_parse * 100:.0f}%",
    ))

    # Accumulate diagnosis hints
    n         = len(rows)
    death_n   = cause_counts.get("death", 0)
    timeout_n = cause_counts.get("timeout", 0)
    evac_n    = cause_counts.get("evacuated", 0)

    if avg_parse < 0.80:
        diagnosis.append(
            f"[{diff}] Low parse rate ({avg_parse*100:.0f}%) — "
            "action schema may be unclear; consider simplifying JSON keys."
        )
    if avg_think < 0.50:
        diagnosis.append(
            f"[{diff}] Low think rate ({avg_think*100:.0f}%) — "
            "model is skipping <think> tags; prompt may need stronger CoT instruction."
        )
    if evac_n == 0 and diff == "easy":
        diagnosis.append(
            "[easy] Zero evacuations on easy difficulty — "
            "check exit reachability, narrative clarity, or BFS distance from spawn."
        )
    if n > 0 and death_n / n > 0.80:
        diagnosis.append(
            f"[{diff}] {death_n}/{n} episodes end in death — "
            "damage rates may be too high or smoke/fire proximity warnings are not clear."
        )
    if n > 0 and timeout_n / n > 0.80:
        diagnosis.append(
            f"[{diff}] {timeout_n}/{n} episodes time out — "
            "model may be looping/waiting; exits might be hard to locate from narratives."
        )
    if avg_step_rew < -0.3 and evac_rate < 0.1:
        diagnosis.append(
            f"[{diff}] Very negative mean step reward ({avg_step_rew:+.3f}) with near-zero "
            "success — model may be actively moving into fire; check DangerPenalty trigger conditions."
        )

print(sep + "\n")

if diagnosis:
    print("DIAGNOSTICS (environment design signals)")
    print("-" * 50)
    for d in diagnosis:
        print(f"  • {d}")
    print()
else:
    print("  No red-flag patterns detected — environment appears legible to this model.\n")

# ── Save CSV ───────────────────────────────────────────────────────────────────
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
model_slug = MODEL_ID.replace("/", "_").replace(".", "-")
csv_path   = Path(OUTPUT_DIR) / f"hf_baseline_{model_slug}_{timestamp}.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "difficulty", "seed", "evacuated", "cause_of_end",
    "final_health", "total_reward", "mean_step_reward",
    "steps_taken", "max_steps",
    "think_rate", "parse_rate", "format_score_avg", "error",
]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)

print(f"CSV saved → {csv_path}")